# Comparing Models for Daily Rat Sightings in New York City

## Import packages

In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

import xgboost as xgb
from xgboost import plot_importance


## Importing the Data

In [21]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Baseline Seasonal Average Model

In [22]:
years_back_use = 4
day_window_use = 4

In [23]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    df["ds"] = pd.to_datetime(df["ds"])
    df["doy"] = df["ds"].dt.dayofyear
    df["year"] = df["ds"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "y"].mean())
    return pd.Series(forecasts, index=target_dates)

In [24]:
results = []
rs["ds"] = pd.to_datetime(rs["ds"])

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["ds"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, 
                                       target_dates=target_dates, 
                                       years_back=years_back_use,
                                       day_window=day_window_use)

    # We take the true values.
    y_true = test["y"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,18.567526,0.251569
1,1,11.781678,0.202731
2,2,12.970780,0.219000
3,3,21.392505,0.237162
4,4,19.552544,0.180788
5,5,27.097225,0.257168
6,6,22.906809,0.217712
7,7,23.659416,0.240364
8,8,15.552538,0.210615
9,9,18.707209,0.207081


## Baseline Year Ago Rolling 4 Week Average 

In [25]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [26]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]
rolling4w_results_df

,fold,rmse,mape
0,0,15.558645,0.245443
1,1,20.800713,0.393455
2,2,14.232759,0.250932
3,3,19.439237,0.224473
4,4,16.407479,0.175315
5,5,21.056896,0.238259
6,6,28.874575,0.359895
7,7,28.131547,0.340850
8,8,17.986850,0.240132
9,9,25.218332,0.233300


## Prophet Model

In [27]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [28]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [29]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet_results_df = pd.DataFrame(results)
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]
prophet_results_df

09:46:11 - cmdstanpy - INFO - Chain [1] start processing
09:46:12 - cmdstanpy - INFO - Chain [1] done processing
09:46:12 - cmdstanpy - INFO - Chain [1] start processing
09:46:12 - cmdstanpy - INFO - Chain [1] done processing
09:46:12 - cmdstanpy - INFO - Chain [1] start processing
09:46:13 - cmdstanpy - INFO - Chain [1] done processing
09:46:13 - cmdstanpy - INFO - Chain [1] start processing
09:46:13 - cmdstanpy - INFO - Chain [1] done processing
09:46:13 - cmdstanpy - INFO - Chain [1] start processing
09:46:14 - cmdstanpy - INFO - Chain [1] done processing
09:46:14 - cmdstanpy - INFO - Chain [1] start processing
09:46:14 - cmdstanpy - INFO - Chain [1] done processing
09:46:14 - cmdstanpy - INFO - Chain [1] start processing
09:46:15 - cmdstanpy - INFO - Chain [1] done processing
09:46:15 - cmdstanpy - INFO - Chain [1] start processing
09:46:15 - cmdstanpy - INFO - Chain [1] done processing
09:46:16 - cmdstanpy - INFO - Chain [1] start processing
09:46:16 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,11.407934,0.156252
1,1,12.729309,0.209772
2,2,10.389350,0.149250
3,3,14.795726,0.145784
4,4,13.396296,0.122439
5,5,18.284586,0.175340
6,6,14.608378,0.136541
7,7,13.302925,0.124126
8,8,13.455639,0.183782
9,9,11.247166,0.125919


## NeuralProphet Model

Since the Prophet model seemed to work so well (see below), we might ask about whether or not one cna improve on it. We try the ['Neural Prophet model'](https://neuralprophet.com/) which should theoretically provide the either the same results or improvements.

Edit: This section is currently commented out and comparisons will be done in another notebook. This is to speed up run time fo this notebook.

In [30]:
# # this is the time series split we will work with
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# # we import the data and clean it for future use
# rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# # mark cutoff dates, and also rename columns
# rs = rs[rs['created_date']<'2025-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [31]:
# from neuralprophet import NeuralProphet

# import numpy as np
# np.NaN = np.nan


# # the following packages are meant to turn off a bunch of the warnings and ERRORs that pop up while running NeuralProphet.
# # the errors that do show up are not all that important and a lot is due to outdated packages.
# import warnings
# import logging

# warnings.filterwarnings("ignore")

# logging.getLogger("neuralprophet").setLevel(logging.ERROR)
# logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
# logging.getLogger("NP").setLevel(logging.ERROR)

In [32]:
# ## Add weather data.

# import requests
# import pandas as pd

# lat, lon = 40.7831, -73.9712
# start = "2020-01-01"
# end   = "2025-02-28" # <"2025-03-01"

# url = (
#     "https://archive-api.open-meteo.com/v1/archive"
#     f"?latitude={lat}&longitude={lon}"
#     f"&start_date={start}&end_date={end}"
#     "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
#     "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
#     "precipitation_sum,snowfall_sum"
#     "&timezone=America/New_York"
# )

# response = requests.get(url)
# data = response.json()

# if 'error' in data:
#     nd = pd.read_csv("weatherdata.csv")
#     nd = nd.set_index('date')
#     wd = nd
    
# else:
#     wd = pd.DataFrame(data["daily"])
#     wd["date"] = pd.to_datetime(wd["time"])
#     wd = wd.set_index("date")

# def add_weather_data_no_index(df,wd):
#     if "time" in wd.columns:
#         wd = wd.drop(columns=["time"])

#     for column in wd.columns:
#         df[column] = wd[column].values

#     return df

In [33]:
# regressed_features = ['apparent_temperature_max', 
#                       'apparent_temperature_min',
#                     'snowfall_sum']
# wd = wd.reset_index(drop=True).rename(columns={"time": "ds"})
# wd["ds"] = pd.to_datetime(wd["ds"])
# rs["ds"] = pd.to_datetime(rs["ds"])

# rs = rs.merge(
#     wd[['ds'] + regressed_features],
#     on="ds",
#     how="left"
# )

# lags_for_regressed_features = dict()
# lags_for_regressed_features['apparent_temperature_max'] = 30
# lags_for_regressed_features['apparent_temperature_min'] = 14
# lags_for_regressed_features['snowfall_sum'] = 3

In [34]:
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):

#     train = rs.iloc[train_index].copy()
#     test = rs.iloc[test_index].copy()

#     train = train.dropna(subset=["y"])


#     model = NeuralProphet(yearly_seasonality=True, 
#                           #weekly_seasonality=True, 
#                           epochs = 25,
#                           accelerator = 'auto',
#                           n_lags=7)

#     model = model.add_country_holidays(country_name="US")

#     for column in regressed_features:
#         model.add_lagged_regressor(column, n_lags=lags_for_regressed_features[column])

#     # merge regressors correctly
#     # train = train.merge(wd[['ds'] + regressed_features], on="ds", how="left")

#     model.fit(train, freq="D", progress="off")

#     # build dataframe containing future regressors
#     future = pd.concat([train[['ds','y'] + regressed_features], test[['ds','y']].merge(wd[['ds'] + regressed_features], on="ds", how="left")])
#     forecast = model.predict(future)

#     y_pred = forecast["yhat1"].iloc[-len(test):].values
#     y_true = test["y"].values

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)

#     results.append({"fold": i, "rmse": rmse, "mape": mape})

# neural_prophet_results_df = pd.DataFrame(results)
# neural_prophet_results_df.loc["mean"] = ["mean", neural_prophet_results_df["rmse"].mean(), neural_prophet_results_df["mape"].mean()]
# neural_prophet_results_df

## SARIMAX Model

The reason why the SARIMA / SARIMAX model does not perform as well as we'd like is discussed here: https://stats.stackexchange.com/questions/613677/using-sarimax-for-daily-data-with-yearly-seasonal-pattern. An excellent read for more details can be found here: https://robjhyndman.com/hyndsight/longseasonality/. For these reasons, instead of using SARIMA's included seasonality features for the yearly seasonality, we instead add Fourier terms as exogeneous variables. Furthermore, we make the point that ARIMA is only good over a short interval. Since our goal is 14 days out, as opposed to 7 days, the predictions are not as good.

In [35]:
from pmdarima import auto_arima
from statsmodels.tsa.statespace.sarimax import SARIMAX


In [36]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [37]:
def fourier_terms(df, periods, n_terms):
    t = np.arange(1, len(df) + 1)
    fourier_df = pd.DataFrame()
    
    for period in periods:
        for i in range(1, n_terms + 1):
            fourier_df[f'{period}sin_{i}'] = np.sin(2 * np.pi * i * t / period)
            fourier_df[f'{period}cos_{i}'] = np.cos(2 * np.pi * i * t / period)
    
    return fourier_df

fourier_train = fourier_terms(rs, [365], 10)
exog = fourier_train

In [38]:
results = []

# Loop through each fold
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    exog_train = exog.iloc[train_index]
    exog_test = exog.iloc[test_index]

    orders = (2,1,1)
    seasonal_orders = (1,0,1,7)

    # we can use auto_arima to get optimal (p, d, q) and (P, D, Q, s) parameters for SARIMAX. just need to uncomment the following code.
    # model_auto = auto_arima(train['y'], 
    #                         exog=exog_train,  # exogenous Fourier terms for training data
    #                         seasonal=True, 
    #                         m=7, #  
    #                         trace=True, 
    #                         stepwise=True,  # Stepwise search to speed up
    #                         suppress_warnings=True, 
    #                         maxiter=300,  # Limit the number of iterations
    #                         max_p=3, 
    #                         max_q=3, 
    #                         max_P=2, 
    #                         max_Q=2, 
    #                         d=1,# might want to tune d 
    #                         D=1 # might want to tune D
    #                         )
    # orders = model_auto.order  # (p, d, q)
    # seasonal_orders = model_auto.seasonal_order  # (P, D, Q, s)
    
    # Fit the SARIMAX model with the exogenous features (Fourier terms)
    model_sarimax = SARIMAX(train['y'], 
                            order=orders,  
                            seasonal_order=seasonal_orders,  
                            exog=exog_train,  # Exogenous Fourier terms for training data
                            )
    
    model_fit = model_sarimax.fit(disp=False)
    
    # Predict for the test period. Have to remember to subtract 1 to get the correct index.
    y_pred = model_fit.predict(start=len(train), end=len(train)+len(test)-1, exog=exog_test, dynamic=False)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

sarima_results_df = pd.DataFrame(results)
sarima_results_df.loc['mean'] = ['mean',  sarima_results_df['rmse'].mean(), sarima_results_df['mape'].mean()]
sarima_results_df

/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
/opt/anaconda3/envs/erdos_ds_environment/lib/python3.12/site-packages/statsmodels/tsa/statespace/sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('No

,fold,rmse,mape
0,0,10.746025,0.148659
1,1,14.826875,0.272614
2,2,10.222825,0.166586
3,3,17.573015,0.177943
4,4,11.672750,0.111016
5,5,17.515287,0.169149
6,6,18.306540,0.222385
7,7,17.661033,0.199586
8,8,18.389369,0.234732
9,9,13.140892,0.111558


## Holt-Winters Model

In [39]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [40]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [41]:
results = []
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    holt_winters = ExponentialSmoothing(train['y'], seasonal='add', seasonal_periods=365).fit(optimized=True)
    
    y_pred = holt_winters.forecast(len(test))
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

hw_results_df = pd.DataFrame(results)
hw_results_df.loc['mean'] = ['mean',  hw_results_df['rmse'].mean(), hw_results_df['mape'].mean()]
hw_results_df

,fold,rmse,mape
0,0,18.504062,0.283771
1,1,19.683080,0.356323
2,2,17.156330,0.287733
3,3,20.420253,0.220282
4,4,19.755392,0.200969
5,5,24.175004,0.246227
6,6,31.776648,0.352027
7,7,27.104526,0.305690
8,8,27.631400,0.392618
9,9,25.827218,0.294849


## MSTL Model

MSTL Model stands for Multiple Seasonal-Trend decomposition using LOESS. For more, see https://nixtlaverse.nixtla.io/statsforecast/docs/models/multipleseasonaltrend.html.

*If the code in this section is currently commented out and one wants to see the results, then one should uncomment and make sure to edit the results table below to include the results of this run.*

In [42]:
# !pip install statsforecast

In [43]:
# from statsforecast import StatsForecast
# from statsforecast.models import MSTL, AutoARIMA

In [44]:
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

# # again, we reimport the data for ease of running

# rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs['closed_date'] = pd.to_datetime(rs['closed_date'])
# rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# # mark cutoff dates, and also rename columns
# rs = rs[rs['created_date']<'2025-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# # This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
# print(len(rs)==2251)


In [45]:
# # specific to Statsforecast requirements

# rs.columns = ['ds', 'y']
# rs.insert(0, 'unique_id', 'DAILY_RAT_SIGHTINGS')
# rs = rs.sort_values(['unique_id', 'ds']).reset_index(drop=True)
# rs.tail()

In [46]:
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# results = []

# def create_dataset(series, time_step=1):
#     X, y = [], []
#     for i in range(len(series) - time_step):
#         X.append(series[i:(i + time_step), 0])
#         y.append(series[i + time_step, 0])
#     return np.array(X), np.array(y)

# time_step = 10
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]

#     series = train['y'].values

#     # scale the data
#     models = [MSTL(
#     season_length=[7, 7*4, 7*52], # seasonalities of the time series (weekly, monthly, yearly)
#     trend_forecaster=AutoARIMA() # model used to forecast trend
#     )]
#     sf = StatsForecast(
#     models=models, # model used to fit
#     freq='d', # frequency of the data
#     )

#     sf = sf.fit(df=rs)
#     forecasts = sf.predict(h=14, level=[90]) # 90 means (confidence percentile) of the prediction interval. 

#     y_pred = forecasts['MSTL'].values
#     y_true = test['y'].values

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)

#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# mstl_results_df = pd.DataFrame(results)
# mstl_results_df.loc['mean'] = ['mean',  mstl_results_df['rmse'].mean(), mstl_results_df['mape'].mean()]
# mstl_results_df

## LSTM Model

LSTM stands for Long Term Short Memory. Here, we simply try to use the LSTM model by itself and check how it performs. For future purposes, it might be of interest to see if a ['hybrid model'](https://peerj.com/articles/cs-1001/#proposed-hybrid-model) using Prophet and LSTM might be able to produce better results.

If the code in this section is currently commented out and one wants to see the results, then one should uncomment and make sure to edit the results table below to include the results of this run.

In [47]:
# pip install neuralforecast

In [48]:
# from neuralforecast import NeuralForecast
# from neuralforecast.models import LSTM
# import pandas as pd
# import numpy as np

In [49]:
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

# # again, we reimport the data for ease of running

# rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs['closed_date'] = pd.to_datetime(rs['closed_date'])
# rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# # mark cutoff dates, and also rename columns
# rs = rs[rs['created_date']<'2025-03-01']
# rs = rs[rs['created_date']>='2020-01-01']
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# # This is should be 2251 which equals the number of days from 2020-01-01 to 2026-02-28
# print(len(rs)==2251)

# rs['ds'] = pd.to_datetime(rs['ds'])
# rs['ds'] = rs['ds'].astype('datetime64[ns]')

In [50]:
# time_step = 14
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):

#     train = rs.iloc[train_index].copy()
#     test = rs.iloc[test_index].copy()

#     # need to do some heavy formatting
#     train_df = train[['ds', 'y']].copy()
#     train_df['unique_id'] = 'series_1'
#     train_df = train_df[['unique_id', 'ds', 'y']]
#     test_df = test[['ds', 'y']].copy()
#     test_df['unique_id'] = 'series_1'
#     test_df = test_df[['unique_id', 'ds', 'y']]

#     horizon = len(test_df)

#     models = [LSTM(h=horizon, max_steps=100, scaler_type='standard')]
#     nf = NeuralForecast(models=models, freq='D')

#     nf.fit(df=train_df)
#     forecasts = nf.predict(h=14)

#     y_pred = forecasts['LSTM'].values
#     y_true = test_df['y'].values

#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)

#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# lstm_results_df = pd.DataFrame(results)
# lstm_results_df.loc['mean'] = ['mean',  lstm_results_df['rmse'].mean(), lstm_results_df['mape'].mean()]
# lstm_results_df

## XGBoost Model

The XGBoost model requires a bit more preparatory work. Our current dataframe rs is quite bare. We will need to add features for use.

In [51]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

### Adding Features to XGBoost

In [52]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    # Must shift by 14 days because we do not want to let there be temporal leakage in our evaluations
    df['ma7'] = df['y'].shift(14).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(14).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(14).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(14).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(14).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(14).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(14).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(14).rolling(window=365).mean()
    
    return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [53]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2025-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [54]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Remove overlapping columns to avoid join errors
    overlap = wd.columns.intersection(df.columns)
    wd = wd.drop(columns=overlap)
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)
    df['apparent_temperature_min_lag18'] = (df.index - pd.Timedelta('18 days')).map(target_map)
    df['apparent_temperature_min_lag19'] = (df.index - pd.Timedelta('19 days')).map(target_map)
    df['apparent_temperature_min_lag20'] = (df.index - pd.Timedelta('20 days')).map(target_map)
    df['apparent_temperature_min_lag21'] = (df.index - pd.Timedelta('21 days')).map(target_map)

    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [55]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [56]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [57]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)

In [58]:
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

rs.columns

Index(['y', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear', 'dayofmonth',
       'weekofyear', 'dayofweek_sin', 'dayofweek_cos', 'month_sin',
       'month_cos', 'lag15', 'lag16', 'lag30', 'lag60', 'lag90', 'lag120',
       'lag150', 'lag180', 'lag362', 'lag363', 'lag364', 'lag365', 'lag366',
       'lag367', 'lag730', 'lag1095', 'lag1460', 'lag1825', 'ma7', 'ma30',
       'ma60', 'ma90', 'ma120', 'ma150', 'ma180', 'ma365',
       'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
       'apparent_temperature_max', 'apparent_temperature_min',
       'apparent_temperature_mean', 'precipitation_sum', 'snowfall_sum',
       'apparent_temperature_min_lag1', 'apparent_temperature_min_lag7',
       'apparent_temperature_min_lag14', 'apparent_temperature_min_lag15',
       'apparent_temperature_min_lag16', 'apparent_temperature_min_lag17',
       'apparent_temperature_min_lag18', 'apparent_temperature_min_lag19',
       'apparent_temperature_min_lag20', 'apparent_tempera

### Features for XGBoost

In [59]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'dayofyear', 'temperature_2m_max_lag14', 'temperature_2m_max_lag30',
            'temperature_2m_max_lag60', 'is_federal_holiday', 
            'lag15', 'lag16', 
            'lag30', 'lag60', 'lag90', 'lag120',
            'lag150', 'lag180', 'lag362', 'lag363', 'lag364', 'lag365', 'lag366',
            'lag367',
            ]

### Parameters for XGBoost

In [60]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': 0.5, 
         'n_estimators': 1000, 
        #  'min_child_weight': 6, 
        # 'learning_rate': 0.001,
        # 'max_depth': 6, 
        # 'subsample': 1,
        # 'colsample_bytree': 0.96,
        # 'colsample_bylevel': 0.6, 
        # 'colsample_bynode': 0.9, 
        # 'reg_alpha': 2.2, 
        # 'gamma': 100, 
        # 'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }

### Results for XGBoost Model

In [61]:
# print(FEATURES)
# print(params)
TARGET = 'y'

# Gotta make sure the features and parameters exist.

reg = xgb.XGBRegressor(**params)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    reg.fit(train[FEATURES], train[TARGET])
    y_pred = reg.predict(test[FEATURES])
    y_true = test[TARGET].values
    
    # Our metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

xgb_results_df = pd.DataFrame(results)
mean_rmse = xgb_results_df['rmse'].mean()
mean_mape = xgb_results_df['mape'].mean()
xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

In [62]:
xgb_results_df

,fold,rmse,mape
0,0,17.864241,0.226696
1,1,24.067712,0.398887
2,2,18.800473,0.315739
3,3,18.655793,0.222418
4,4,15.654417,0.156093
5,5,21.615169,0.206902
6,6,20.530798,0.219343
7,7,22.760787,0.187518
8,8,22.758990,0.292473
9,9,14.384402,0.164482


## XGBoosted Prophet Model

In [63]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [64]:
def add_new_lags(df, x):
    # lags
    target_map = df[x].to_dict()
    df[f'{x}lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag16'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag17'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag18'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag19'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag20'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag21'] = (df.index - pd.Timedelta('16 days')).map(target_map)

    df[f'{x}lag30'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    df[f'{x}lag365'] = (df.index - pd.Timedelta('16 days')).map(target_map)
    df[f'{x}lag730'] = (df.index - pd.Timedelta('15 days')).map(target_map)
    return df

In [65]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'dayofyear', 'temperature_2m_max_lag14', 'temperature_2m_max_lag30',
            'temperature_2m_max_lag60', 
            'is_federal_holiday', 
            'lag15', 'lag16', 'lag30', 'lag60', 'lag90', 'lag120', 'lag150', 
            'lag180', 
            'lag362', 'lag363', 'lag364', 'lag365', 'lag366', 'lag367',
            'residualslag15', 'residualslag16', 'residualslag17',
            'residualslag18', 'residualslag19', 'residualslag20', 'residualslag21',
            'residualslag30', 'residualslag365', 'residualslag730', 
            'trend', 'yhat_lower', 'yhat_upper', 
            ]

### Parameters for XGB+Prophet

In [66]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
        # 'base_score': 0.5, 
        # 'n_estimators': 2000, 
        #  'min_child_weight': 6, 
        # 'learning_rate': 0.01,
        # 'max_depth': 6, 
        # 'subsample': 1,
        # 'colsample_bytree': 0.96,
        # 'colsample_bylevel': 0.6, 
        # 'colsample_bynode': 0.9, 
        # 'reg_alpha': 2.2, 
        # 'gamma': 100, 
        # 'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }

In [67]:
date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [68]:
save = rs['ds'].copy().values
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)
rs['ds']=save
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')
rs

,y,ds,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,...,apparent_temperature_min_lag365,apparent_temperature_min_lag730,temperature_2m_max_lag14,temperature_2m_max_lag30,temperature_2m_max_lag60,is_federal_holiday,Trash_Law,New_Trash_Law,Rat_Mitigation_Zone,Rat_Czar_Appointed
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2020-01-01,2,1,1,2020,1,1,1,0.974928,...,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0
2020-01-02,40,2020-01-02,3,1,1,2020,2,2,1,0.433884,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-03,41,2020-01-03,4,1,1,2020,3,3,1,-0.433884,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-04,25,2020-01-04,5,1,1,2020,4,4,1,-0.974928,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-05,17,2020-01-05,6,1,1,2020,5,5,1,-0.781831,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-02-24,57,2025-02-24,0,1,2,2025,55,24,9,0.000000,...,-11.1,-10.8,0.2,-2.2,-0.2,0,1,1,1,1
2025-02-25,61,2025-02-25,1,1,2,2025,56,25,9,0.781831,...,-6.9,-8.1,-0.4,4.1,2.5,0,1,1,1,1
2025-02-26,61,2025-02-26,2,1,2,2025,57,26,9,0.974928,...,-4.1,-7.1,0.6,2.0,7.5,0,1,1,1,1


In [69]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset into training and testing sets
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet on the training data
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Make predictions on the training set to calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
    train_forecast = model.predict(train_future)
    
    # Calculate residuals (actual - predicted) on the training data
    train_residuals = train['y'].values - train_forecast['yhat'].values
    
#    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    # Build a new DataFrame of residuals
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

    train['residuals']=train_residuals
    add_new_lags(train, 'residuals')

    train['trend'] = train_forecast['trend'].values
    train['yhat_lower'] = train_forecast['yhat_lower'].values
    train['yhat_upper'] = train_forecast['yhat_upper'].values
    
    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_residuals, y_train_residuals)

    test['residuals'] = np.nan
    
    dummy = pd.concat([X_train_residuals, test], axis=0)  # row-wise   
    add_new_lags(dummy,'residuals')

    test = dummy.iloc[test_index]

    # Forecast using Prophet on the test set
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)

    
    # Predict residuals using XGBoost for the test set
    test['trend'] = prophet_forecast[-len(test):]['trend'].values
    test['yhat_lower'] = prophet_forecast[-len(test):]['yhat_lower'].values
    test['yhat_upper'] = prophet_forecast[-len(test):]['yhat_upper'].values
    
    X_test = test[FEATURES]  # Features for the test set
    xgb_residual_preds = xgb_model.predict(X_test)
    
    
    
    # Combine Prophet's forecast and XGBoost's residual prediction
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Store the results for this fold
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
    # Uncomment code below if you want to have plots on feature importance. I'll leave it commented out for obvious reasons.
    
    # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
    # plot_importance(xgb_model, ax=ax1, importance_type='gain')
    # ax1.set_title('Gain-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax2, importance_type='weight')
    # ax2.set_title('Split-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax3, importance_type='cover')
    # ax3.set_title('Cover Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
    # ax4.set_title('Total Gain Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
    # ax5.set_title('Total Cover Importance', fontsize=12)

    plt.show()

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

09:50:22 - cmdstanpy - INFO - Chain [1] start processing
09:50:22 - cmdstanpy - INFO - Chain [1] done processing
/var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_74398/2858050151.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train['residuals']=train_residuals
/var/folders/ry/m6r2ndwd10bdv8tvww5hr2680000gn/T/ipykernel_74398/1320145536.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[f'{x}lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
/var/folders/ry/m6r2ndwd10bdv8tvww5h

In [70]:
prophet_xgb_results_df

,fold,rmse,mape
0,0,11.947081,0.163684
1,1,12.144424,0.226971
2,2,9.587388,0.141368
3,3,16.074790,0.159605
4,4,14.747463,0.142854
5,5,21.060712,0.192953
6,6,13.345548,0.138557
7,7,11.607957,0.084979
8,8,14.958357,0.166629
9,9,11.689559,0.124906


# Conclusions on Model Comparisons

## Results Table

In [71]:
# We make a dictionary of models and their results to make it easier to iterate over. 
# Make sure to add to this if writing new models in.
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
    'prophet': prophet_results_df,
    'sarima': sarima_results_df,
    'hw': hw_results_df,
    'xgb': xgb_results_df,
    'prophet+xgb': prophet_xgb_results_df,
    #'LSTM': lstm_results_df,
    #'MSTL': mstl_results_df,
    #'neural_prophet': neural_prophet_results_df
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                          \
model   baseline         hw    prophet prophet+xgb  rolling4w     sarima   
fold                                                                       
0      18.567526  18.504062  11.407934   11.947081  15.558645  10.746025   
1      11.781678  19.683080  12.729309   12.144424  20.800713  14.826875   
2      12.970780  17.156330  10.389350    9.587388  14.232759  10.222825   
3      21.392505  20.420253  14.795726   16.074790  19.439237  17.573015   
4      19.552544  19.755392  13.396296   14.747463  16.407479  11.672750   
5      27.097225  24.175004  18.284586   21.060712  21.056896  17.515287   
6      22.906809  31.776648  14.608378   13.345548  28.874575  18.306540   
7      23.659416  27.104526  13.302925   11.607957  28.131547  17.661033   
8      15.552538  27.631400  13.455639   14.958357  17.986850  18.389369   
9      18.707209  25.827218  11.247166   11.689559  25.218332  13.140892   
10     24.272366  27.674471  17.168479   16.291460  23.559006  19.581356   
11     18.153566  26.248850  12.378810   14.903157  30.471151  13.421587   
12     24.853346  28.760867  14.563015   15.087246  22.437214  14.778049   
13     16.828983  20.550270  12.273410   18.380241  20.442209  17.710563   
14     22.776938  22.675944  13.236741   16.024296  20.328332  12.315364   
15     18.884245  20.803197  11.470989   13.530421  24.711731  11.863642   
16     14.104677  22.297709  14.869562   14.389749  14.372282  15.491332   
17     12.975088  19.135250  10.308464    8.489035  15.811106   9.616687   
18     12.866083  16.131421   9.081193   15.203159  11.038488  12.236973   
19     12.724520  17.508347  11.471978   12.476631  21.875459  13.033949   
20     13.934647  14.995134   9.810502   10.499899  11.189520   8.481183   
21     15.114747  12.677572  19.276898   17.471048  23.220796  17.096622   
22     19.059046  18.632907  17.160892   18.628634  14.059821  10.465981   
23     12.462791  12.471724  13.402697   10.422871  14.528298  10.740365   
24     13.023250  14.535294  10.282737    9.828528  12.220811   6.931098   
25     12.671733  16.210605   9.087091   10.948095  13.113515   7.846857   
mean   17.572856  20.897826  13.056183   13.836067  19.272568  13.525624   

                      mape                                            \
model        xgb  baseline        hw   prophet prophet+xgb rolling4w   
fold                                                                   
0      17.864241  0.251569  0.283771  0.156252    0.163684  0.245443   
1      24.067712  0.202731  0.356323  0.209772    0.226971  0.393455   
2      18.800473  0.219000  0.287733  0.149250    0.141368  0.250932   
3      18.655793  0.237162  0.220282  0.145784    0.159605  0.224473   
4      15.654417  0.180788  0.200969  0.122439    0.142854  0.175315   
5      21.615169  0.257168  0.246227  0.175340    0.192953  0.238259   
6      20.530798  0.217712  0.352027  0.136541    0.138557  0.359895   
7      22.760787  0.240364  0.305690  0.124126    0.084979  0.340850   
8      22.758990  0.210615  0.392618  0.183782    0.166629  0.240132   
9      14.384402  0.207081  0.294849  0.125919    0.124906  0.233300   
10     19.876968  0.191193  0.226430  0.134008    0.132421  0.182605   
11     23.665246  0.225834  0.320476  0.153072    0.182128  0.389008   
12     17.207280  0.249034  0.272964  0.115936    0.110404  0.247137   
13     19.319529  0.179169  0.199836  0.136457    0.186115  0.231404   
14     17.821696  0.239011  0.261001  0.104961    0.122359  0.237294   
15     21.121208  0.208619  0.246600  0.128084    0.180503  0.344475   
16     20.905273  0.225494  0.323921  0.210380    0.198106  0.231666   
17     14.193466  0.209072  0.272619  0.154340    0.126018  0.261849   
18     12.915066  0.175885  0.230804  0.146453    0.210547  0.172352   
19     22.348518  0.348903  0.461033  0.250069    0.288030  0.625055   
20     14.198694  0.359298  0.370644  0.219563    0.215551  0.275160   
21 

## Summary

In the above table, we see that the NeruralProphet / Prophet model had the best average RMSE. We also see that the next best were the SARIMA model and the Prophet + XGBoost models. Therefore, we will select the Prophet model for our modeling purposes and only maybe consider training a Prophet + XGBoost model to see if there is any improvement. We now know that the Prophet model performs the best, but we might ask if using an XGBoost model to predict residuals can lead to better model performance. In the above, we were unable to do that, but we were also using suboptimal parameters. 


Cautionary tales have been told about NeuralProphet and Prophet. For this reason, despite its performance here, we will be very careful to make sure that indeed these are the best models for the data available.

In [72]:
# We export the final table to save for later.
# Set a title 
#final_table = final_table.style.set_caption('NYC: Comparison of Models using Walk-Forward Validation with 23 folds and 14 Day Test Sizes')
#final_table.to_csv('citywide_comparison_results.csv', index=True)

# Prophet Models with varying degrees of Initial Data

One might ask if there is too much noise at the start of the data. Afterall, 2020-2021 were still during the years where the effects of the COVID-19 pandemic were being felt. We consider what happens to Prophet's predictions as we drop data before 2020, 2021, 2022, 2023, and 2024. Of course, dropping data before 2020 just corresponds to the results above. We collect the RMSEs and MAPEs into a table and display the results in a markdown cell below at the end of this section. As a preview, we do not end up finding significant improvements in the model by dropping dates before a given year.

## Dropping <2020 data

In [73]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2020-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 since there was a leap year
print(len(rs)==2251-366)

date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet20_results_df = pd.DataFrame(results)
prophet20_results_df.loc['mean'] = ['mean',  prophet20_results_df['rmse'].mean(), prophet20_results_df['mape'].mean()]
prophet20_results_df

09:50:55 - cmdstanpy - INFO - Chain [1] start processing


False


09:50:55 - cmdstanpy - INFO - Chain [1] done processing
09:50:56 - cmdstanpy - INFO - Chain [1] start processing
09:50:56 - cmdstanpy - INFO - Chain [1] done processing
09:50:56 - cmdstanpy - INFO - Chain [1] start processing
09:50:56 - cmdstanpy - INFO - Chain [1] done processing
09:50:57 - cmdstanpy - INFO - Chain [1] start processing
09:50:57 - cmdstanpy - INFO - Chain [1] done processing
09:50:57 - cmdstanpy - INFO - Chain [1] start processing
09:50:57 - cmdstanpy - INFO - Chain [1] done processing
09:50:58 - cmdstanpy - INFO - Chain [1] start processing
09:50:58 - cmdstanpy - INFO - Chain [1] done processing
09:50:58 - cmdstanpy - INFO - Chain [1] start processing
09:50:58 - cmdstanpy - INFO - Chain [1] done processing
09:50:59 - cmdstanpy - INFO - Chain [1] start processing
09:50:59 - cmdstanpy - INFO - Chain [1] done processing
09:50:59 - cmdstanpy - INFO - Chain [1] start processing
09:50:59 - cmdstanpy - INFO - Chain [1] done processing
09:51:00 - cmdstanpy - INFO - Chain [1] 

,fold,rmse,mape
0,0,11.407934,0.156252
1,1,12.729309,0.209772
2,2,10.389350,0.149250
3,3,14.795726,0.145784
4,4,13.396296,0.122439
5,5,18.284586,0.175340
6,6,14.608378,0.136541
7,7,13.302925,0.124126
8,8,13.455639,0.183782
9,9,11.247166,0.125919


## Dropping <2021 data

In [74]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2021-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 since there was a leap year
print(len(rs)==2251-366)

date_range = pd.date_range(start="2021-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet21_results_df = pd.DataFrame(results)
prophet21_results_df.loc['mean'] = ['mean',  prophet21_results_df['rmse'].mean(), prophet21_results_df['mape'].mean()]
prophet21_results_df

09:51:11 - cmdstanpy - INFO - Chain [1] start processing
09:51:11 - cmdstanpy - INFO - Chain [1] done processing


False


09:51:11 - cmdstanpy - INFO - Chain [1] start processing
09:51:11 - cmdstanpy - INFO - Chain [1] done processing
09:51:12 - cmdstanpy - INFO - Chain [1] start processing
09:51:12 - cmdstanpy - INFO - Chain [1] done processing
09:51:12 - cmdstanpy - INFO - Chain [1] start processing
09:51:12 - cmdstanpy - INFO - Chain [1] done processing
09:51:13 - cmdstanpy - INFO - Chain [1] start processing
09:51:13 - cmdstanpy - INFO - Chain [1] done processing
09:51:13 - cmdstanpy - INFO - Chain [1] start processing
09:51:13 - cmdstanpy - INFO - Chain [1] done processing
09:51:14 - cmdstanpy - INFO - Chain [1] start processing
09:51:14 - cmdstanpy - INFO - Chain [1] done processing
09:51:15 - cmdstanpy - INFO - Chain [1] start processing
09:51:15 - cmdstanpy - INFO - Chain [1] done processing
09:51:16 - cmdstanpy - INFO - Chain [1] start processing
09:51:16 - cmdstanpy - INFO - Chain [1] done processing
09:51:16 - cmdstanpy - INFO - Chain [1] start processing
09:51:16 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,11.033292,0.151625
1,1,16.152778,0.274071
2,2,13.721482,0.211463
3,3,13.118181,0.131267
4,4,12.344274,0.107165
5,5,18.443387,0.180348
6,6,13.953504,0.127777
7,7,12.577090,0.127647
8,8,13.113848,0.171501
9,9,11.646891,0.127050


## Dropping <2022 data

In [75]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2022-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365)

date_range = pd.date_range(start="2022-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet22_results_df = pd.DataFrame(results)
prophet22_results_df.loc['mean'] = ['mean',  prophet22_results_df['rmse'].mean(), prophet22_results_df['mape'].mean()]
prophet22_results_df

09:51:25 - cmdstanpy - INFO - Chain [1] start processing
09:51:25 - cmdstanpy - INFO - Chain [1] done processing


False


09:51:25 - cmdstanpy - INFO - Chain [1] start processing
09:51:25 - cmdstanpy - INFO - Chain [1] done processing
09:51:25 - cmdstanpy - INFO - Chain [1] start processing
09:51:25 - cmdstanpy - INFO - Chain [1] done processing
09:51:26 - cmdstanpy - INFO - Chain [1] start processing
09:51:26 - cmdstanpy - INFO - Chain [1] done processing
09:51:26 - cmdstanpy - INFO - Chain [1] start processing
09:51:26 - cmdstanpy - INFO - Chain [1] done processing
09:51:26 - cmdstanpy - INFO - Chain [1] start processing
09:51:26 - cmdstanpy - INFO - Chain [1] done processing
09:51:27 - cmdstanpy - INFO - Chain [1] start processing
09:51:27 - cmdstanpy - INFO - Chain [1] done processing
09:51:27 - cmdstanpy - INFO - Chain [1] start processing
09:51:27 - cmdstanpy - INFO - Chain [1] done processing
09:51:27 - cmdstanpy - INFO - Chain [1] start processing
09:51:27 - cmdstanpy - INFO - Chain [1] done processing
09:51:28 - cmdstanpy - INFO - Chain [1] start processing
09:51:28 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,10.648234,0.147345
1,1,15.224251,0.254414
2,2,11.665969,0.172612
3,3,12.885666,0.137431
4,4,11.771681,0.107720
5,5,17.778088,0.173191
6,6,14.756907,0.150087
7,7,12.843009,0.127815
8,8,16.189486,0.214198
9,9,12.110537,0.138533


## Dropping <2023 data

In [76]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2023-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365-365)

date_range = pd.date_range(start="2023-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet23_results_df = pd.DataFrame(results)
prophet23_results_df.loc['mean'] = ['mean',  prophet23_results_df['rmse'].mean(), prophet23_results_df['mape'].mean()]
prophet23_results_df

09:51:36 - cmdstanpy - INFO - Chain [1] start processing
09:51:36 - cmdstanpy - INFO - Chain [1] done processing


False


09:51:36 - cmdstanpy - INFO - Chain [1] start processing
09:51:36 - cmdstanpy - INFO - Chain [1] done processing
09:51:36 - cmdstanpy - INFO - Chain [1] start processing
09:51:36 - cmdstanpy - INFO - Chain [1] done processing
09:51:36 - cmdstanpy - INFO - Chain [1] start processing
09:51:36 - cmdstanpy - INFO - Chain [1] done processing
09:51:37 - cmdstanpy - INFO - Chain [1] start processing
09:51:37 - cmdstanpy - INFO - Chain [1] done processing
09:51:37 - cmdstanpy - INFO - Chain [1] start processing
09:51:37 - cmdstanpy - INFO - Chain [1] done processing
09:51:37 - cmdstanpy - INFO - Chain [1] start processing
09:51:37 - cmdstanpy - INFO - Chain [1] done processing
09:51:37 - cmdstanpy - INFO - Chain [1] start processing
09:51:37 - cmdstanpy - INFO - Chain [1] done processing
09:51:38 - cmdstanpy - INFO - Chain [1] start processing
09:51:38 - cmdstanpy - INFO - Chain [1] done processing
09:51:38 - cmdstanpy - INFO - Chain [1] start processing
09:51:38 - cmdstanpy - INFO - Chain [1]

,fold,rmse,mape
0,0,18.552095,0.250740
1,1,11.803929,0.185149
2,2,8.867677,0.130611
3,3,23.709932,0.263858
4,4,21.627837,0.226241
5,5,24.100218,0.228280
6,6,15.384690,0.124908
7,7,15.451277,0.161402
8,8,18.972768,0.265241
9,9,11.409716,0.131477


## Dropping <2024 data

In [77]:
# this is the time series split we will work with
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)


# we import the data and clean it for future use
rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])
# mark cutoff dates, and also rename columns
rs = rs[rs['created_date']<'2025-03-01']
rs = rs[rs['created_date']>='2024-01-01']
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# This is should be 2251 - 366 - 365 since there was a leap year
print(len(rs)==2251-366-365-365-366)

date_range = pd.date_range(start="2024-01-01", end="2025-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet24_results_df = pd.DataFrame(results)
prophet24_results_df.loc['mean'] = ['mean',  prophet24_results_df['rmse'].mean(), prophet24_results_df['mape'].mean()]
prophet24_results_df

09:51:43 - cmdstanpy - INFO - Chain [1] start processing
09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:43 - cmdstanpy - INFO - Chain [1] start processing


False


09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:43 - cmdstanpy - INFO - Chain [1] start processing
09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:43 - cmdstanpy - INFO - Chain [1] start processing
09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:43 - cmdstanpy - INFO - Chain [1] start processing
09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:43 - cmdstanpy - INFO - Chain [1] start processing
09:51:43 - cmdstanpy - INFO - Chain [1] done processing
09:51:44 - cmdstanpy - INFO - Chain [1] start processing
09:51:44 - cmdstanpy - INFO - Chain [1] done processing
09:51:44 - cmdstanpy - INFO - Chain [1] start processing
09:51:44 - cmdstanpy - INFO - Chain [1] done processing
09:51:44 - cmdstanpy - INFO - Chain [1] start processing
09:51:44 - cmdstanpy - INFO - Chain [1] done processing
09:51:44 - cmdstanpy - INFO - Chain [1] start processing
09:51:44 - cmdstanpy - INFO - Chain [1] done processing
09:51:44 - cmdstanpy - INFO - Chain [1] 

,fold,rmse,mape
0,0,10.937939,0.148404
1,1,15.397247,0.278976
2,2,9.619536,0.164645
3,3,21.436810,0.223148
4,4,18.027758,0.184243
5,5,18.568780,0.171830
6,6,19.011233,0.231564
7,7,18.866084,0.227145
8,8,25.023987,0.357328
9,9,12.967084,0.147490


## Results Table and Conclusion

The results displayed below show that including pre-2020 does not drastically improve Prophet's forecasting performance.

In [78]:
# We make a dictionary of models and their results to make it easier to iterate over. 
# Make sure to add to this if writing new models in.
models = {
    'prophet24': prophet24_results_df,
    'prophet23': prophet23_results_df,
    'prophet22': prophet22_results_df,
    'prophet21': prophet21_results_df,
    'prophet20': prophet20_results_df,
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                  mape  \
model  prophet20  prophet21  prophet22  prophet23  prophet24 prophet20   
fold                                                                     
0      11.407934  11.033292  10.648234  18.552095  10.937939  0.156252   
1      12.729309  16.152778  15.224251  11.803929  15.397247  0.209772   
2      10.389350  13.721482  11.665969   8.867677   9.619536  0.149250   
3      14.795726  13.118181  12.885666  23.709932  21.436810  0.145784   
4      13.396296  12.344274  11.771681  21.627837  18.027758  0.122439   
5      18.284586  18.443387  17.778088  24.100218  18.568780  0.175340   
6      14.608378  13.953504  14.756907  15.384690  19.011233  0.136541   
7      13.302925  12.577090  12.843009  15.451277  18.866084  0.124126   
8      13.455639  13.113848  16.189486  18.972768  25.023987  0.183782   
9      11.247166  11.646891  12.110537  11.409716  12.967084  0.125919   
10     17.168479  14.609297  14.968674  15.411180  15.389093  0.134008   
11     12.378810  13.237088  15.785363  17.389125  19.090649  0.153072   
12     14.563015  14.180012  12.568711  13.329467  13.041634  0.115936   
13     12.273410  12.871856  12.110207  17.339478  14.941627  0.136457   
14     13.236741  12.728129  13.059333  16.638128  13.973282  0.104961   
15     11.470989  11.303323  12.894022  23.860535  18.811281  0.128084   
16     14.869562  14.261721  12.533111  29.559379  22.410296  0.210380   
17     10.308464  11.112866  10.075020  25.792919  17.592807  0.154340   
18      9.081193   9.182969   9.732298  18.581575   9.756483  0.146453   
19     11.471978  11.167771  11.650968  28.501123  18.034895  0.250069   
20      9.810502  10.232086   9.178971  18.034949   9.477889  0.219563   
21     19.276898  19.409104  19.504853  19.106858  14.387921  0.647867   
22     17.160892  16.026082  16.758802  16.718198  18.692646  0.376276   
23     13.402697  12.156412  13.280051  13.683685  19.927053  0.251313   
24     10.282737   8.865842  12.345700  10.892439  14.957078  0.203399   
25      9.087091   7.292101   7.997885   5.868317  21.882512  0.180590   
mean   13.056183  12.874669  13.089146  17.714904  16.623985  0.190076   

                                               
model prophet21 prophet22 prophet23 prophet24  
fold                                           
0      0.151625  0.147345  0.250740  0.148404  
1      0.274071  0.254414  0.185149  0.278976  
2      0.211463  0.172612  0.130611  0.164645  
3      0.131267  0.137431  0.263858  0.223148  
4      0.107165  0.107720  0.226241  0.184243  
5      0.180348  0.173191  0.228280  0.171830  
6      0.127777  0.150087  0.124908  0.231564  
7      0.127647  0.127815  0.161402  0.227145  
8      0.171501  0.214198  0.265241  0.357328  
9      0.127050  0.138533  0.131477  0.147490  
10     0.122117  0.123946  0.125865  0.126678  
11     0.163081  0.196256  0.217854  0.241396  
12     0.114931  0.112105  0.149981  0.145638  
13     0.145823  0.138267  0.180376  0.159198  
14     0.096423  0.105107  0.208529  0.163417  
15     0.129579  0.151355  0.324008  0.251696  
16     0.194868  0.168855  0.466248  0.339418  
17     0.162180  0.147696  0.441631  0.290881  
18     0.150117  0.157504  0.311126  0.138082  
19     0.258251  0.265953  0.719899  0.454995  
20     0.225929  0.207933  0.446556  0.188785  
21     0.631671  0.617281  0.639768  0.471481  
22     0.353309  0.363463  0.363310  0.375325  
23     0.227078  0.248375  0.256963  0.397604  
24     0.173547  0.246488  0.221906  0.343338  
25     0.136087  0.153055  0.096762  0.472811  
mean   0.188266  0.193346  0.274565  0.257520

The results above above indicate that the Prophet model *does not* on average improve significantly as we start dropping data before a given year. We can see this in some folds e.g. fold 0, 1, 2. We also see that there are some folds where the RMSE basically stays the same. Then there are folds like folds 15 and 11 where things seem to improve by dropping some years. The best average RMSE, however, does occur if we drop all pre-2022 data though the improvement is very minor. One might hypothesize that this was due to the lfiting of COVID-19 lockdowns.

# Comparing Prophet Model versus an XGBoosted Prophet Model

This section of code can be long so we summarize. First, we saw from before that Prophet performed very well. We also saw that Prophet + XGBoost on residuals performed slightly worse. This could be due to not tuning the parameters very well. So, we might instead try to see if we can get better predictions by actually doing some feature engineering and hyperparameter tuning of XGBoost.

There are a few things we must be careful about.

1. Remember that we are evaluating based on walk-forward validation. That means we fix a training set, and then evaluate 14 days out. Then for the next fold, we add those 14 days to our training set and evaluate 14 days out again. Then add those 14 days to the training set and evaluate 14 days out again. Then we repeat until we have done 26 folds.

2. Suppose we fix a fold so we have a training set and a test set decided. We first fit Prophet to the training set. Then determine the residuals there. Using the residuals, we then train an XGBoost model fitted to the residuals. Then, we use both the Prophet model and the XGBoost model to forecast 14 days out and we evaluate against the test set.

3. We want **clear** evidence of improvement. So, we had proceeded as before and used a TimeSeriesSplit. In training the XGBoost model, we need to be careful of overfitting. We use Optuna to tune the hyperparameters and manually select features until we see improvement. The benefit of using Optuna is that we have written our objective function to avoid overfitting to the residuals' training set. 

4. Even if we have *clear* evidence of improvement, we also must ensure that the added **complexity** is worth it. Training an XGBoost model is quite time intensive and having to tune hyperparameters for each fold is already a lot of work. The value of Optuna here is that we can limit the number of trials.

## Reimport the Data

We reimport the data so that we can focus only on running this section's code as opposed to having to rerun from the start.

In [79]:
# tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

In [80]:
# rs = pd.read_csv('../../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
# rs['created_date'] = pd.to_datetime(rs['created_date']) 
# rs['closed_date'] = pd.to_datetime(rs['closed_date'])
# rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])

In [81]:
# # Start by cutting off data before 2020-01-01 and after 2026-02-28.
# rs = rs[rs['created_date']<'2025-03-01']
# rs = rs[rs['created_date']>='2020-01-01']

In [82]:
# rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')

In [83]:
# ## This is 2251 which equals the number of days from 2020-01-01 to 2026-02-28
# len(rs)

In [84]:
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Repeat the Computations for Prophet

In [85]:
# # Create a date range covering 2020 through end of 2025
# date_range = pd.date_range(start="2020-01-01", end="2025-02-28")

# # Generate US federal holidays
# calendar = USFederalHolidayCalendar()
# holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# # Build the DataFrame in the same structure as your original
# federal_holidays = pd.DataFrame({
#     'holiday': 'federal_us',
#     'ds': pd.to_datetime(holidays),
#     'lower_window': 0,
#     'upper_window': 1,
# })

# holidays = federal_holidays

# # Rename columns for Prophet model
# rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     model = Prophet(holidays=holidays)
#     model.add_country_holidays(country_name='US')

#     model.fit(train)
    
#     future = model.make_future_dataframe(periods=len(test), freq='D')
#     forecast = model.predict(future)
    
#     # Obtain predicted values and compare against the actuals.
#     y_pred = forecast['yhat'][-len(test):].values
#     y_true = test['y'].values
    
#     # Calculate RMSE
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
#     # Calculate MAPE
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     # Append results
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# # Convert results to a datafrane
# prophet_results_df = pd.DataFrame(results)

# prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

## Adding Features to XGBoost

In [86]:
# def create_features(df):
#     """
#     Create time series features based on time series index.
#     """
#     df = df.copy()
#     df['dayofweek'] = df.index.dayofweek
#     df['quarter'] = df.index.quarter
#     df['month'] = df.index.month
#     df['year'] = df.index.year
#     df['dayofyear'] = df.index.dayofyear
#     df['dayofmonth'] = df.index.day
#     df['weekofyear'] = df.index.isocalendar().week
#     return df

# def add_cyclic(df):
#     target_map = df['y'].to_dict()
#     df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
#     df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
#     df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
#     df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
#     return df

# def add_lags(df):
#     target_map = df['y'].to_dict()
#     df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
#     df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
#     df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
#     df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
#     df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
#     df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
#     df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
#     df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
#     df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
#     df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
#     df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
#     df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
#     df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
#     df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
#     return df

# def add_seasonal_lags(df):
#     target_map = df['y'].to_dict()
#     df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
#     df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
#     df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
#     df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
#     df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
#     df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

#     df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
#     df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
#     df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
#     df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
#     df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
#     return df

# def add_moving_averages(df):
#     df = df.copy()
    
#     # Ensure sorted by date
#     df = df.sort_index()
    
#     # Moving averages (using previous values only)
#     df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
#     df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
#     df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
#     df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
#     df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
#     df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
#     df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
#     df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
#     return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [87]:
# ## Add weather data.

# import requests
# import pandas as pd

# lat, lon = 40.7831, -73.9712
# start = "2020-01-01"
# end   = "2025-02-28"

# url = (
#     "https://archive-api.open-meteo.com/v1/archive"
#     f"?latitude={lat}&longitude={lon}"
#     f"&start_date={start}&end_date={end}"
#     "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
#     "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
#     "precipitation_sum,snowfall_sum"
#     "&timezone=America/New_York"
# )

# response = requests.get(url)
# data = response.json()

# if 'error' in data:
#     nd = pd.read_csv("weatherdata.csv")
#     nd = nd.set_index('date')
#     wd = nd

# else:
#     wd = pd.DataFrame(data["daily"])
#     wd["date"] = pd.to_datetime(wd["time"])
#     wd = wd.set_index("date")

In [88]:
# def add_weather_data(df, wd):
#     df = df.copy()
#     wd = wd.copy()
    
#     # Ensure datetime index
#     df.index = pd.to_datetime(df.index)
#     wd.index = pd.to_datetime(wd.index)
    
#     # Drop unnecessary column
#     if "time" in wd.columns:
#         wd = wd.drop(columns=["time"])
    
#     # Join on date index
#     df = df.join(wd, how="left")
    
#     return df

In [89]:
# from pandas.tseries.holiday import USFederalHolidayCalendar

# def add_federal_holidays(df, custom_holidays=None):
#     df = df.copy()
    
#     # Ensure datetime index
#     df.index = pd.to_datetime(df.index)
    
#     cal = USFederalHolidayCalendar()
#     holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
#     if custom_holidays:
#         for d in custom_holidays:
#             if len(d) == 5:  # MM-DD format → recurring annually
#                 years = df.index.year.unique()
#                 for y in years:
#                     holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
#             else:  # YYYY-MM-DD → one specific date
#                 holidays = holidays.append(pd.to_datetime([d]))
    
#     holidays = holidays.drop_duplicates().sort_values()
    
#     df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
#     return df

In [90]:
# def add_law_flag(df, law_name: str, start_date: str):
#     # Adds a binary column to indicate when a new law is active.
#     df = df.copy()
    
#     # Ensure datetime index
#     df.index = pd.to_datetime(df.index)
    
#     # Convert start_date to datetime
#     start_dt = pd.to_datetime(start_date)
    
#     # Create binary column: 1 if date >= start_date, else 0
#     df[law_name] = (df.index >= start_dt).astype(int)
    
#     return df

In [91]:
# # This must be run importing weather data

# def add_more_weather_feature(df):
#     target_map = df['apparent_temperature_min'].to_dict()
#     df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
#     df['apparent_temperature_min_lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
#     df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    
#     df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
#     df['apparent_temperature_min_lag15'] = (df.index - pd.Timedelta('15 days')).map(target_map)
#     df['apparent_temperature_min_lag16'] = (df.index - pd.Timedelta('16 days')).map(target_map)
#     df['apparent_temperature_min_lag17'] = (df.index - pd.Timedelta('17 days')).map(target_map)

#     df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
#     df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
#     df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
#     df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
#     df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
#     df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
#     df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
#     df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
#     df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
#     df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
#     df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
#     df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
#     df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
#     df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

#     target_map = df['temperature_2m_max'].to_dict()
#     df['temperature_2m_max_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
#     df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
#     df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
#     df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

#     return df

In [92]:
# def add_forecast_features(df,forecast):
#     for column in forecast.columns:
#         df[column] = forecast[column]

#     return df

In [93]:
# forecast

## Features for XGBoost

In [94]:
# FIRST_FEATURES = ['apparent_temperature_min_lag1',
#             'temperature_2m_max_lag1',
#             'dayofyear', 
#             'temperature_2m_max_lag60'
#             ]

## Add default parameters for XGBoost

In [95]:
# params = {'objective': 'reg:squarederror',
#          'eval_metric': 'rmse',
#          'booster': 'gbtree',
#          'base_score': -5, 
#          'n_estimators': 3000, 
#          'min_child_weight': 5, 
#          'learning_rate': 0.001,
#          'max_depth': 7,
#          #'subsample': 0.9,
#          #'colsample_bytree': 0.7,
#          #'colsample_bylevel': 0.6, 
#          #'colsample_bynode': 0.9, 
#          #'reg_alpha': 4, 
#          #'gamma': 1.5, 
#          #'reg_lambda': 4.5,
#          #'early_stopping_rounds': 100, 
#         }

In [96]:
## Old code to evaluate XGBoost + Prophet model.

# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     # Split the dataset into training and testing sets
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     # Fit Prophet on the training data
#     model = Prophet(holidays=holidays)
#     model.add_country_holidays(country_name='US')
#     model.fit(train)
    
#     # Make predictions on the training set to calculate residuals
#     train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
#     train_forecast = model.predict(train_future)
    
#     # Calculate residuals (actual - predicted) on the training data
#     train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
#     # Build a new DataFrame of residuals
#     residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

#     train = train.set_index('ds')
#     train.index = pd.to_datetime(train.index)
#     train = create_features(train)
#     train = add_cyclic(train)
#     train = add_lags(train)
#     train = add_seasonal_lags(train)
#     train = add_moving_averages(train)
#     train = add_weather_data(train,wd)
#     train = add_more_weather_feature(train)
#     train = add_federal_holidays(train, custom_holidays = ['12-31'])
#     train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
#     train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     X_train_residuals = train[FEATURES]
#     y_train_residuals = residuals_df['y']

    
    
#     xgb_model = xgb.XGBRegressor(**params)
#     xgb_model.fit(X_train_residuals, y_train_residuals)
    

#     test = test.set_index('ds')
#     test.index = pd.to_datetime(test.index)
#     test = create_features(test)
#     test = add_cyclic(test)
#     test = add_lags(test)
#     test = add_seasonal_lags(test)
#     test = add_moving_averages(test)
#     test = add_weather_data(test,wd)
#     test = add_more_weather_feature(test)
#     test = add_federal_holidays(test, custom_holidays = ['12-31'])
#     test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
#     test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     # Predict residuals using XGBoost for the test set
#     X_test = test[FEATURES]  # Features for the test set
#     xgb_residual_preds = xgb_model.predict(X_test)
    
#     # Forecast using Prophet on the test set
#     future = model.make_future_dataframe(periods=len(test), freq='D')
#     prophet_forecast = model.predict(future)
    
#     # Combine Prophet's forecast and XGBoost's residual prediction
#     y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    
#     # Actual values for the test set
#     y_true = test['y'].values
    
#     # Calculate RMSE
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
#     # Calculate MAPE
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     # Store the results for this fold
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
#     ## Uncomment code below if you want to have plots on feature importance.
#     # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
#     # plot_importance(xgb_model, ax=ax1, importance_type='gain')
#     # ax1.set_title('Gain-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax2, importance_type='weight')
#     # ax2.set_title('Split-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax3, importance_type='cover')
#     # ax3.set_title('Cover Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
#     # ax4.set_title('Total Gain Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
#     # ax5.set_title('Total Cover Importance', fontsize=12)

    
#     plt.show()

# # Convert the results into a DataFrame
# prophet_xgb_results_df = pd.DataFrame(results)

# prophet_xgb_results_df = pd.DataFrame(results)
# mean_rmse = prophet_xgb_results_df['rmse'].mean()
# mean_mape = prophet_xgb_results_df['mape'].mean()
# prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

## Optuna tuning before each fold for best hyperparameters

The following is a very long code block and probably takes quite a long time to run so I've tried my best to break down the workings of the code with tons of comments.

In [97]:
# import optuna
# from optuna.exceptions import TrialPruned
# from sklearn.model_selection import train_test_split

# results = []

# # def objective(trial, X_train_residuals, y_train_residuals):
# #     param = {'objective': 'reg:squarederror',
# #          'eval_metric': 'rmse',
# #          'booster': 'gbtree',
# #         'n_estimators': trial.suggest_int('n_estimators', 700, 1000),
# #         'max_depth': trial.suggest_int('max_depth', 1, 5),
# #         'learning_rate': trial.suggest_float('learning_rate', 0.01, 2, log=True),
# #         'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
# #         'subsample': trial.suggest_float('subsample', 0.5, 1.0),
# #         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
# #         'gamma': trial.suggest_float('gamma', 0, 5),
# #         'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
# #         'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
# #         'random_state': 42
# #     }

# #     tscv_inner = TimeSeriesSplit(n_splits=5)  # small CV on fold

# #     rmses = []
# #     for tr_idx, val_idx in tscv_inner.split(X_train_residuals):
# #         X_tr, X_val = X_train_residuals.iloc[tr_idx], X_train_residuals.iloc[val_idx]
# #         y_tr, y_val = y_train_residuals.iloc[tr_idx], y_train_residuals.iloc[val_idx]

# #         model = xgb.XGBRegressor(**param)
# #         model.fit(X_tr, y_tr)
# #         y_pred = model.predict(X_val)
# #         rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

# #     return np.mean(rmses)

# def objective(trial, X_train_residuals, y_train_residuals):
#     param = {'objective': 'reg:squarederror',
#              'eval_metric': 'rmse',
#              'booster': 'gbtree',
#              'n_estimators': trial.suggest_int('n_estimators', 700, 1000),
#              'max_depth': trial.suggest_int('max_depth', 1, 5),
#              'learning_rate': trial.suggest_float('learning_rate', 0.01, 2, log=True),
#              'min_child_weight': trial.suggest_int('min_child_weight', 4, 10),
#              'subsample': trial.suggest_float('subsample', 0.5, 1.0),
#              'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
#              'gamma': trial.suggest_float('gamma', 0, 5),
#              'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
#              'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
#              'random_state': 42
#     }

#     tscv_inner = tscv
#     rmses = []

#     for fold, (tr_idx, val_idx) in enumerate(tscv_inner.split(X_train_residuals), 1):
#         X_tr, X_val = X_train_residuals.iloc[tr_idx], X_train_residuals.iloc[val_idx]
#         y_tr, y_val = y_train_residuals.iloc[tr_idx], y_train_residuals.iloc[val_idx]

#         model = xgb.XGBRegressor(**param)
#         model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

#         y_pred = model.predict(X_val)
#         fold_rmse = np.sqrt(mean_squared_error(y_val, y_pred))
#         rmses.append(fold_rmse)

#         trial.report(np.mean(rmses), fold)
#         if trial.should_prune():
#             raise TrialPruned(f"Trial was pruned at fold {fold}.")

#     return np.mean(rmses)

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     # Split the dataset
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     # Fit Prophet
#     model = Prophet(holidays=holidays)
#     model.add_country_holidays(country_name='US')
#     model.fit(train)
    
#     # Calculate residuals
#     train_future = model.make_future_dataframe(periods=0, freq='D')
#     train_forecast = model.predict(train_future)
#     train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
#     residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals})
    
#     # Feature engineering
#     train = train.set_index('ds')
#     train.index = pd.to_datetime(train.index)
#     train = create_features(train)
#     train = add_cyclic(train)
#     train = add_lags(train)
#     train = add_seasonal_lags(train)
#     train = add_moving_averages(train)
#     train = add_weather_data(train, wd)
#     train = add_more_weather_feature(train)
#     train = add_federal_holidays(train, custom_holidays=['12-31'])
#     train = add_law_flag(train, law_name='Trash_Law', start_date='2024-03-01')
#     train = add_law_flag(train, law_name='New_Trash_Law', start_date='2024-11-01')
#     train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
#     train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date='2023-04-12')
#     train = add_forecast_features(train, train_forecast)

#     FEATURES = FIRST_FEATURES + list(train_forecast.columns)
#     FEATURES = [x for x in FEATURES if x not in ['ds', 'Christmas Day',
#     'Christmas Day_lower', 'Christmas Day_upper', 'Christmas Day (observed)',
#     'Christmas Day (observed)_lower', 'Christmas Day (observed)_upper',
#     'Columbus Day', 'Columbus Day_lower', 'Columbus Day_upper',
#     'Independence Day', 'Independence Day_lower', 'Independence Day_upper', 'Independence Day (observed)','Independence Day (observed)_lower', 'Independence Day (observed)_upper',
#     'Juneteenth National Independence Day', 'Juneteenth National Independence Day_lower', 'Juneteenth National Independence Day_upper', 
#     'Juneteenth National Independence Day (observed)', 'Juneteenth National Independence Day (observed)_lower', 'Juneteenth National Independence Day (observed)_upper',
#     'Labor Day', 'Labor Day_lower', 'Labor Day_upper',
#     'Martin Luther King Jr. Day', 'Martin Luther King Jr. Day_lower', 'Martin Luther King Jr. Day_upper',
#     'Memorial Day', 'Memorial Day_lower', 'Memorial Day_upper',
#     "New Year's Day", "New Year's Day_lower", "New Year's Day_upper",
#     "New Year's Day (observed)", "New Year's Day (observed)_lower", "New Year's Day (observed)_upper",
#     'Thanksgiving Day', 'Thanksgiving Day_lower', 'Thanksgiving Day_upper',
#     'Veterans Day', 'Veterans Day_lower', 'Veterans Day_upper',
#     'Veterans Day (observed)', 'Veterans Day (observed)_lower', 'Veterans Day (observed)_upper',
#     "Washington's Birthday", "Washington's Birthday_lower", "Washington's Birthday_upper",
#     'federal_us', 'federal_us_lower', 'federal_us_upper',
#     'holidays', 'holidays_lower', 'holidays_upper',
#     'multiplicative_terms', 'multiplicative_terms_lower', 'multiplicative_terms_upper']]

#     X_train_residuals = train[FEATURES]
#     y_train_residuals = residuals_df['y']

#     best_params = params
#     # Uncomment the next 4 lines to include Optuna hyperparameter tuning. We used Optuna because our parameter space is quite large.
#     # study = optuna.create_study(direction='minimize')
#     # study.optimize(lambda trial: objective(trial, X_train_residuals, y_train_residuals), n_trials=10)
#     # best_params = study.best_params 
#     # best_params['random_state'] = 42

#     # Train XGBoost with best parameters from Optuna.
#     xgb_model = xgb.XGBRegressor(**best_params)
#     xgb_model.fit(X_train_residuals, y_train_residuals)
    
#     # Add the features we will use for XGBoost. Again, we are using XGBoost to try and predict the residuals of the Prophet model.
#     test = test.set_index('ds')
#     test.index = pd.to_datetime(test.index)
#     test = create_features(test)
#     test = add_cyclic(test)
#     test = add_lags(test)
#     test = add_seasonal_lags(test)
#     test = add_moving_averages(test)
#     test = add_weather_data(test, wd)
#     test = add_more_weather_feature(test)
#     test = add_federal_holidays(test, custom_holidays=['12-31'])
#     test = add_law_flag(test, law_name='Trash_Law', start_date='2024-03-01')
#     test = add_law_flag(test, law_name='New_Trash_Law', start_date='2024-11-01')
#     test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
#     test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date='2023-04-12')


    
#     # Have prophet make a forecast on the test set which is 14 days.
#     future = model.make_future_dataframe(periods=len(test), freq='D')
#     prophet_forecast = model.predict(future)
#     test = add_forecast_features(test, prophet_forecast)

#     # Predict the residuals of Prophet for test set
#     X_test = test[FEATURES]
#     xgb_residual_preds = xgb_model.predict(X_test)
    
#     # Add the forecast of the residuals by XGBoost for 14 days.
#     y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
#     y_true = test['y'].values

#     # Obtain metrics for our forecast against the actuals.
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# # Results DataFrame
# prophet_xgb_results_df = pd.DataFrame(results)
# mean_rmse = prophet_xgb_results_df['rmse'].mean()
# mean_mape = prophet_xgb_results_df['mape'].mean()
# prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

In [98]:
# # here were the features we were using. tune them!
# FEATURES

In [99]:
# X_train_residuals.plot(figsize=(24, 6))

In [100]:
# y_train_residuals.plot(figsize=(24, 6))

In [101]:
## Uncomment code below if you want to have plots on feature importance.
## if one runs this after the previous code block, you're getting feature importance of the last step of walk-forward validation

# fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
# plot_importance(xgb_model, ax=ax1, importance_type='gain')
# ax1.set_title('Gain-based Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax2, importance_type='weight')
# ax2.set_title('Split-based Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax3, importance_type='cover')
# ax3.set_title('Cover Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
# ax4.set_title('Total Gain Importance', fontsize=12)

# plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
# ax5.set_title('Total Cover Importance', fontsize=12)

In [102]:
# display(prophet_xgb_results_df)
# display(prophet_results_df)